In [1]:
devtools::load_all("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/gwastools")
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/")
library(data.table)
library(digest)

i Loading gwastools

Loading required package: data.table

Loading required package: ggplot2

Loading required package: stringr



In [2]:
args <- list(
    min_mac = 4,
    phenotype = "COPD_combined_primary_care",
    chromosome = "chr1",
    path_markers="data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense_markers.txt.gz",
    path_ac_by_phenotypes="data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense_AC.txt.gz",
    path_hash_by_phenotypes="data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense_hash.txt.gz",
    path_markers_by_gene="data/conditional/rare/combined/genes/min_mac4/ukb_eur_wes_200k_maf0to5e-2_COPD_combined_primary_care_pLoF_damaging_missense.txt.gz"

)

In [3]:
print(args)

stopifnot(file.exists(args$path_markers))
stopifnot(file.exists(args$path_ac_by_phenotypes))
stopifnot(file.exists(args$path_hash_by_phenotypes))
stopifnot(file.exists(args$path_markers_by_gene))

$min_mac
[1] 4

$phenotype
[1] "COPD_combined_primary_care"

$chromosome
[1] "chr1"

$path_markers
[1] "data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense_markers.txt.gz"

$path_ac_by_phenotypes
[1] "data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense_AC.txt.gz"

$path_hash_by_phenotypes
[1] "data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense_hash.txt.gz"

$path_markers_by_gene
[1] "data/conditional/rare/combined/genes/min_mac4/ukb_eur_wes_200k_maf0to5e-2_COPD_combined_primary_care_pLoF_damaging_missense.txt.gz"



In [4]:
filter_to_candidates <- function(dt, pheno, chr, csqs = c("pLoF", "damaging_missense"), verbose = TRUE){
    
    #' @param dt a data.frame with the columns, rsid, phenotype and consequence_category
    #' @param pheno phenotype to filter on
    #' @param chr chromosome to filter on
    #' @param csqs consequences to filter on (vector)
    
    # check input
    autosomes <- paste0("chr",1:22)
    stopifnot(chr %in% autosomes)
    stopifnot("rsid" %in% colnames(dt))
    stopifnot("phenotype" %in% colnames(dt))
    stopifnot("consequence_category" %in% colnames(dt))
    stopifnot(pheno %in% dt$phenotype)
    
    # build chromosome column
    dt$chr <- unlist(
        lapply(dt$rsid, function(x) unlist(strsplit(x, split = ':'))[1])
    )
    
    stopifnot(dt$chr %in% autosomes)
    n0 <- nrow(dt)
    
    # perform subsetting
    dt <- dt[dt$chr %in% chr,]
    dt <- dt[dt$phenotype %in% phenotype,]
    dt <- dt[dt$consequence_category %in% csqs,]
    
    # spit out some summaries
    n1 <- nrow(dt)
    n_genes <- length(unique(dt$ensembl_gene_id))
    msg <- paste("Note: Subsetting to", n0, "markers near", n_genes, "relevant gene(s) on", chr)
    if (verbose) write(msg, stderr())
        
    # change column names
    colnames(dt)[colnames(dt)=="rsid"] <- "id"
        
    return(dt)
    
}

In [5]:
annotate_by_allele_count <- function(dt, phenotype, min_mac){
    
    #' @param dt a data.table with columns, id, phenotypes where
    # each phenotype cell has the allele count for the phenotype
    #' @param phenotype phenotype column that is in dt
    #' @param min_mac integer above zero 
    
    # check input  
    stopifnot(phenotype %in% colnames(dt))
    stopifnot("id" %in% colnames(dt))
    stopifnot(min_mac >= 0)
    cols <- c("id", phenotype)
    
    
    # perform subset
    dt <- dt[,cols, with = FALSE]
    dt$keep_variant_ac <- dt[[phenotype]] >=  min_mac
    #dt <- dt[ d,]
    
    # change colnames
    colnames(dt)[1:2] <- c("id", "ac")
    dt$min_mac_filter <- min_mac
    
    return(dt)
    
}

In [6]:
annotate_by_perfect_ld <- function(dt, phenotype){
    
    #' @param dt a data.table with columns, id, phenotypes where
    # each phenotype cell is the hash of the dosage string
    #' @param phenotype phenotype column that is in dt
    
    # Prune markers in perfect LD
    stopifnot(phenotype %in% colnames(dt))
    stopifnot("id" %in% colnames(dt))
    cols <- c("id", phenotype)
    
    # perform subsets
    dt <- dt[,cols, with = FALSE]
    colnames(dt) <- c("id","hash")
    
    # find markers in perfect LD
    hash_duplicated <- duplicated(dt$hash)
    hash_in_perfect_ld <- dt$hash[hash_duplicated ]
    id_in_perfect_ld <- dt$id[dt$hash %in% hash_in_perfect_ld]
    n <- sum(hash_duplicated)
    
    if (n>0){
        
        # make mapping to markers in perfect LD
        lst <- lapply(hash_in_perfect_ld, function(hash){
            paste(dt$id[ dt$hash %in% hash], collapse = ';')
        })

        print(lst)

        hash_to_markers <- unlist(lst)
        names(hash_to_markers) <- hash_in_perfect_ld

        # filter to markers that are not in perfect LD
        dt$keep_variant_ld <- !hash_duplicated

        # keep track of the markers in perfect LD
        dt$perfect_ld_marker <- hash_to_markers[dt$hash]
        dt$perfect_ld_marker <- unlist(
            lapply(1:nrow(dt), function(idx){
                regex <- paste0(";?",dt$id[idx],";?")
                gsub(regex, "", dt$perfect_ld_marker[idx])
            })
        )
    } else {
        
        dt$keep_variant_ld <- TRUE
        dt$perfect_ld_marker <- NA
        
    }
    
    return(dt)
     
}

In [7]:
chromosome <- args$chromosome
phenotype <- args$phenotype
min_mac <- as.numeric(args$min_mac)

d_gene <- fread(args$path_markers_by_gene)
d_phenos <- fread(args$path_ac_by_phenotypes)
d_hash <- fread(args$path_hash_by_phenotypes)
d_marker <- fread(args$path_markers)



In [8]:
table(d_gene$phenotype)


COPD_combined_primary_care 
                      3047 

In [9]:
# filter to candidate variants 
d_gene_filter <- filter_to_candidates(d_gene, args$phenotype, args$chromosome)
markers_in_gene <- d_gene_filter$id

In [10]:
# filter hash and phenotype markers based on markers in gene
d_phenos <- d_phenos[d_phenos$id %in% markers_in_gene,]
d_hash <- d_hash[d_hash$id %in% markers_in_gene,]

In [11]:
# annotate the data by various metrics
d_ac_filter <- annotate_by_allele_count(d_phenos, args$phenotype, min_mac = 4)
d_ld_filter <- annotate_by_perfect_ld(d_hash, args$phenotype)

In [12]:
# merge the two filters
mrg <- merge(d_ac_filter, d_ld_filter, by = 'id')
stopifnot(nrow(mrg) == nrow(d_ac_filter))
stopifnot(nrow(mrg) == nrow(d_ld_filter))
print(nrow(mrg))
print(length(unique(mrg$id)))

[1] 48
[1] 48


In [13]:
# merge with gene ID and consequence. 

## Note, that the rsids may be duplicated if the same variant affects a two overlapping genes, e.g:
# chr19:40778389:A:G	ENSG00000167578	damaging_missense	COPD_combined_primary_care	chr19
# chr19:40778389:A:G	ENSG00000171570	damaging_missense	COPD_combined_primary_care	chr19
mrg <- merge(mrg, d_gene_filter, by = 'id', all.x = TRUE)
print(nrow(mrg))
print(length(unique(mrg$id)))

[1] 93
[1] 48


In [14]:
mrg$keep <- mrg$keep_variant_ac & mrg$keep_variant_ld
mrg <- mrg[mrg$keep,]
mrg

id,ac,keep_variant_ac,min_mac_filter,hash,keep_variant_ld,perfect_ld_marker,ensembl_gene_id,consequence_category,phenotype,chr,keep
<chr>,<dbl>,<lgl>,<dbl>,<chr>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<lgl>
chr19:40778389:A:G,4,TRUE,4,d702a3a0,TRUE,NA,ENSG00000167578,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40778389:A:G,4,TRUE,4,d702a3a0,TRUE,NA,ENSG00000171570,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780034:T:G,25,TRUE,4,5f727244,TRUE,NA,ENSG00000167578,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780034:T:G,25,TRUE,4,5f727244,TRUE,NA,ENSG00000171570,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780072:C:G,119,TRUE,4,f8cc03a0,TRUE,NA,ENSG00000167578,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780072:C:G,119,TRUE,4,f8cc03a0,TRUE,NA,ENSG00000171570,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780429:C:T,11,TRUE,4,2157c489,TRUE,NA,ENSG00000167578,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780429:C:T,11,TRUE,4,2157c489,TRUE,NA,ENSG00000171570,damaging_missense,COPD_combined_primary_care,chr19,TRUE
chr19:40780441:G:A,5,TRUE,4,d3028d64,TRUE,NA,ENSG00000167578,damaging_missense,COPD_combined_primary_care,chr19,TRUE


In [16]:
# get counts
n_markers_start <- length(unique(d_gene_filter$id))
n_markers_end <-  length(unique(mrg$id))
n_genes <- length(unique(d_gene_filter$ensembl_gene_id))
n_ac_ok <- sum(d_ac_filter$keep_variant_ac)
n_ld_ok <- sum(!d_ld_filter$keep_variant_ld)
n_dup_mark <- ifelse(length(unique(mrg$id)) != length(mrg$id),"Yes","No")
n_ac_above_zero <- sum(d_ac_filter$ac > 0)
n_ac_singletons <- sum(d_ac_filter$ac == 1)

In [17]:
msg <- paste0(
    "# Phenotype: ", phenotype, "\n",
    "# Significant gene(s): ", n_genes, "\n",
    "# Potential hail markers in gene(s): ", n_markers_start, "\n",
    "# Same marker in different gene(s): ", n_dup_mark, "\n",
    "# Markers in perfect LD: ", n_ld_ok, "\n",
    "# Markers with AC>=0: ", n_ac_above_zero, "\n",
    "# Markers with AC==1: ", n_ac_singletons, "\n",
    "# Markers with AC>=", min_mac,": ", n_ac_ok, "\n",
    "# Markers (post filtering of AC>=",min_mac," and LD) count: ", n_markers_end, "\n"
)

write(msg, stdout())

# Phenotype: COPD_combined_primary_care
# Significant gene(s): 3
# Potential hail markers in gene(s): 447
# Same marker in different gene(s): Yes
# Markers in perfect LD: 0
# Markers with AC>=0: 48
# Markers with AC==1: 21
# Markers with AC>=4: 19
# Markers (post filtering of AC>=4 and LD) count: 19



In [29]:
markers

[1] "chr19:40778389:A:G" "chr19:40780034:T:G" "chr19:40780072:C:G"
 [4] "chr19:40780429:C:T" "chr19:40780441:G:A" "chr19:40780447:G:A"
 [7] "chr19:40780499:G:A" "chr19:40783786:C:T" "chr19:40783801:G:A"
[10] "chr19:40783815:G:A" "chr19:40783833:A:C" "chr19:40783988:G:A"
[13] "chr19:40784031:G:A" "chr19:40784063:G:A" "chr19:40784073:A:G"
[16] "chr19:40786700:G:A" "chr19:40786733:C:G" "chr19:40786733:C:T"
[19] "chr19:40786752:T:C"

In [24]:
markers <- mrg$id[mrg$keep]
markers <- markers[!duplicated(markers)]

In [28]:
regex <- paste0(paste0("(",markers,")"), collapse = '|')

In [36]:
vcf_path <- "data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense.vcf.bgz"
cmd <- paste("zcat",vcf_path,"| grep -v '##' | grep -E '", regex,"'" )
cmd
d <- fread(cmd = cmd)

[1] "zcat data/conditional/rare/combined/ukb_eur_wes_200k_chr19_maf0to5e-2_pLoF_damaging_missense.vcf.bgz | grep -v '##' | grep -E ' (chr19:40778389:A:G)|(chr19:40780034:T:G)|(chr19:40780072:C:G)|(chr19:40780429:C:T)|(chr19:40780441:G:A)|(chr19:40780447:G:A)|(chr19:40780499:G:A)|(chr19:40783786:C:T)|(chr19:40783801:G:A)|(chr19:40783815:G:A)|(chr19:40783833:A:C)|(chr19:40783988:G:A)|(chr19:40784031:G:A)|(chr19:40784063:G:A)|(chr19:40784073:A:G)|(chr19:40786700:G:A)|(chr19:40786733:C:G)|(chr19:40786733:C:T)|(chr19:40786752:T:C) '"

In [38]:
head(d)

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V176915,V176916,V176917,V176918,V176919,V176920,V176921,V176922,V176923,V176924
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr19,40780034,chr19:40780034:T:G,T,G,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr19,40780034,chr19:40780034:T:G,T,G,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr19,40780034,chr19:40780034:T:G,T,G,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr19,40780072,chr19:40780072:C:G,C,G,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr19,40780072,chr19:40780072:C:G,C,G,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr19,40780429,chr19:40780429:C:T,C,T,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [ ]:
mrg$keep_variant_ld <- NULL
mrg$perfect_ld_marker <- NULL
mrg$keep_variant_ac <- NULL
head(mrg)

In [209]:
mrg$id[mrg$keep_variant_ac & mrg$keep_variant_ld]

[1] "chr1:45330557:C:A"   "chr1:45331193:C:T"   "chr1:45331228:G:A"  
 [4] "chr1:45331327:T:C"   "chr1:45331529:G:A"   "chr1:45331556:C:T"  
 [7] "chr1:45331558:T:C"   "chr1:45331699:AG:A"  "chr1:45331745:TG:T" 
[10] "chr1:45331771:G:A"   "chr1:45331772:G:T"   "chr1:45332080:G:A"  
[13] "chr1:45332164:ACT:A" "chr1:45332239:G:C"   "chr1:45332278:C:T"  
[16] "chr1:45332279:G:A"   "chr1:45332445:C:T"   "chr1:45332455:C:A"  
[19] "chr1:45332455:C:T"   "chr1:45332479:C:T"   "chr1:45332585:C:T"  
[22] "chr1:45332594:C:T"   "chr1:45332597:T:C"   "chr1:45332614:C:T"  
[25] "chr1:45332615:G:A"   "chr1:45332768:G:A"   "chr1:45332771:C:G"  
[28] "chr1:45332794:C:T"   "chr1:45332795:G:A"   "chr1:45332803:T:C"  
[31] "chr1:45333165:C:T"   "chr1:45333168:A:T"   "chr1:45333171:C:T"  
[34] "chr1:45333294:C:T"   "chr1:45333436:G:A"   "chr1:45333452:C:T"

In [204]:
mrg[mrg$keep_variant_ac > 0 & mrg$keep_variant_ld == TRUE,]

id,ac,keep_variant_ac,min_mac_filter,hash,keep_variant_ld,perfect_ld_marker
<chr>,<dbl>,<lgl>,<dbl>,<chr>,<lgl>,<chr>
chr1:45330557:C:A,20,TRUE,4,53a3d39c,TRUE,NA
chr1:45331193:C:T,13,TRUE,4,47c8312d,TRUE,NA
chr1:45331228:G:A,8,TRUE,4,f2c7db66,TRUE,NA
chr1:45331327:T:C,17,TRUE,4,073e49d1,TRUE,NA
chr1:45331529:G:A,24,TRUE,4,9dc51a39,TRUE,NA
chr1:45331556:C:T,2193,TRUE,4,328bfd72,TRUE,NA
chr1:45331558:T:C,52,TRUE,4,0d705cbe,TRUE,NA
chr1:45331699:AG:A,33,TRUE,4,3eb8fc70,TRUE,NA
chr1:45331745:TG:T,8,TRUE,4,2d2779bb,TRUE,NA


In [30]:

stopifnot(d_gene$chr %in% autosomes)

In [31]:

markers <- intersect(d_marker$rsid, d_gene$rsid)
n_genes <- length(unique(d_gene$ensembl_gene_id))
n_gene_markers <- length(markers)
head(d_gene)

rsid,ensembl_gene_id,consequence_category,phenotype,chr
<chr>,<chr>,<chr>,<chr>,<chr>
chr1:45329315:TG:T,ENSG00000132781,pLoF,CC_combined_primary_care,chr1
chr1:45329439:T:C,ENSG00000132781,pLoF,CC_combined_primary_care,chr1
chr1:45330515:C:A,ENSG00000132781,pLoF,CC_combined_primary_care,chr1
chr1:45330551:G:A,ENSG00000132781,damaging_missense,CC_combined_primary_care,chr1
chr1:45330557:C:A,ENSG00000132781,damaging_missense,CC_combined_primary_care,chr1
chr1:45331175:GTAGTGCC:G,ENSG00000132781,pLoF,CC_combined_primary_care,chr1


In [17]:
### filter by markers near "causal" genes ###





# setup dict
dict_csqs <- d_gene$rsid
names(dict_csqs) <- d_gene$consequence_category


write(msg, stderr())

In [18]:
head(d_gene)

rsid,ensembl_gene_id,consequence_category,phenotype,chr
<chr>,<chr>,<chr>,<chr>,<chr>


In [ ]:
main <- function(args){







  

  ### filter by allele count ###




  ### filter by perfect LD ###

  

  if (length(markers) > 0) {
    # ensure variants follow input formatting required by saige
    markers <- gwastools::order_markers(markers)
    fwrite(data.table(x=markers), args$outfile, row.names = FALSE, col.names = FALSE)
  } else {
    write("No markers present after subsetting", stderr())
  }

}

# add arguments
parser <- ArgumentParser()
parser$add_argument("--chromosome", default=NULL, required = TRUE, help = "Chromosome GRCh38, e.g. chr12")
parser$add_argument("--phenotype", default=NULL, required = TRUE, help = "phenotype, single string.")
parser$add_argument("--path_markers", default=NULL, required = TRUE, help = "list of currently used variants seperated by comma")
parser$add_argument("--path_ac_by_phenotypes", default=NULL, required = TRUE, help = "file to allele count by phenotypes")
parser$add_argument("--path_hash_by_phenotypes", default=NULL, required = TRUE, help = "file containing dosage hashes for removing markers in perfect LD")
parser$add_argument("--path_markers_by_gene", default=NULL, required = TRUE, help = "File containing markers by gene and chromosome")
parser$add_argument("--min_mac", default=1, required = FALSE, help = "Allele count threshold, greater than or equal '>='")
parser$add_argument("--outfile", default=NULL, required = TRUE, help = "where should the subsetted markeres be written")
parser$add_argument("--annotation", default=NULL, required = TRUE, help = "Annotations to perform subset on")
args <- parser$parse_args()